# 01 — Cleaning & EDA

Walking through Online Retail II → bronze/silver/gold.

The full pipeline is in `python/01_build_medallion.py`. This notebook is just so I can see the KPIs and a few cuts without re-running everything.

**What I'm checking**
- Do revenue / orders / customers match `reports/kpi_summary.json`?
- Do November peaks show up in the monthly chart?


In [ ]:
from pathlib import Path
import pandas as pd
import json

ROOT = Path("..")
GOLD = ROOT / "data" / "gold"
kpi = json.loads((ROOT / "reports" / "kpi_summary.json").read_text())
monthly = pd.read_csv(GOLD / "mart_monthly.csv")

print("Period:", kpi["period_start"], "→", kpi["period_end"])
print("Revenue £", round(kpi["revenue"], 2))
print("Orders", kpi["orders"], " Customers", kpi["customers"], " AOV", round(kpi["aov"], 2))
print("UK share", round(kpi["uk_share"], 4), " Return rate", round(kpi["return_rate"], 4))
monthly.tail()

## Cleaning rules (short version)

- Drop exact dupes after unioning the two UCI sheets
- Sales = non-cancel, qty>0, price>0, non-blank description, not a fee code
- Returns kept separately for the return-rate KPI
- Guests (null CustomerID) stay in sales facts but are excluded from RFM

TODO: write a one-pager on fee codes (POST/DOT/M/…) so I don't forget why they're filtered.


In [ ]:
countries = pd.read_csv(GOLD / "mart_country.csv")
products = pd.read_csv(GOLD / "mart_top_products.csv")
print("Top 5 countries by revenue")
display(countries.head())
print("Top 5 products by revenue")
display(products.head())

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(monthly["YearMonth"], monthly["Revenue"] / 1e6, marker="o", ms=3)
ax.set_ylabel("Revenue (£M)")
ax.set_title("Monthly revenue — Nov peaks show up clearly")
ax.tick_params(axis="x", labelrotation=45, labelsize=7)
fig.tight_layout()
plt.show()